In [38]:
import requests
import pandas as pd
import time

# GitHub API 인증 토큰 및 헤더 (실제 유효한 토큰으로 교체 필요)
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN" # 사용자 토큰을 사용합니다.
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}

# 검색을 위한 기본 필터 조건
# 🚨 주의: pushed:2022-01-01..2024-12-31 필터는 현재 2025년 5월 기준으로,
# 2025년에 업데이트된 저장소를 제외합니다. 이로 인해 결과가 0으로 나올 수 있습니다.
# 필요시 이 필터를 조정하세요. (예: pushed:>=2022-01-01 또는 pushed:2022-01-01..{오늘날짜})
BASE_FILTERS = "in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31"

# 검색할 키워드 목록 정의 (키워드 자체에 큰따옴표 제거)
# 검색할 키워드 목록 정의 (키워드 자체에 큰따옴표 제거)
plain_keywords_no_quotes = [

    '"artificial intelligence" OR "computer vision" OR "deep learning" OR "machine learning" OR "image recognition"',
    '"natural language processing" OR "nlp" OR "language model" OR "speech recognition"',
]

ai_adjectives = [
    "agentic", "causal", "explainable", "generative", "multi-modal",
    "quantum", "reliable", "responsible", "robotics", "symbolic"
]


combined_ai_keywords_no_quotes = []
for adj in ai_adjectives:
    combined_ai_keywords_no_quotes.append(f'{adj} AI') # 예: agentic AI (따옴표 없음)

keywords_list_for_query = []
for kw in plain_keywords_no_quotes:
    keywords_list_for_query.append([kw])
for kw in combined_ai_keywords_no_quotes:
    keywords_list_for_query.append([kw])

results_summary = []
total_repos_aggregated = 0 # 각 쿼리 결과의 총합을 저장할 변수

print(f"총 {len(keywords_list_for_query)}개의 키워드 쿼리를 실행합니다.")
print(f"사용될 기본 필터: {BASE_FILTERS}\n")
print("참고: 키워드 내 큰따옴표가 제거되어, 여러 단어 키워드는 AND 조건으로 검색됩니다.")
print("(예: 'artificial intelligence'는 'artificial' AND 'intelligence'로 검색)\n")

for i, group in enumerate(keywords_list_for_query):
    keyword_to_search = group[0]
    full_query = f"{keyword_to_search} {BASE_FILTERS}"
    
    url = "https://api.github.com/search/repositories"
    params = {
        "q": full_query,
        "per_page": 1
    }

    print(f"({i+1}/{len(keywords_list_for_query)}) 실행 중인 쿼리: {full_query}")

    try:
        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()
        count = data.get("total_count", 0)
        
        print(f"  → 결과: {count}개 저장소")
        
        if isinstance(count, int): # count가 정수일 때만 합산
            total_repos_aggregated += count
        
        repo_dict_entry = {keyword_to_search: count}
        results_summary.append(repo_dict_entry)

    except requests.exceptions.HTTPError as http_err:
        error_message = f"HTTP 오류: {http_err}"
        try:
            error_data = response.json()
            error_message += f" - {error_data.get('message', '')}"
            if 'errors' in error_data:
                for err_detail in error_data['errors']:
                    error_message += f" ({err_detail.get('resource', '')} {err_detail.get('field', '')} {err_detail.get('code', '')})"
        except ValueError:
            error_message += f" (응답 내용: {response.text})"
        print(f"  → ❌ {error_message}")
        results_summary.append({keyword_to_search: f"오류 ({response.status_code if response else 'N/A'})"})

    except requests.exceptions.RequestException as req_err:
        print(f"  → ❌ 요청 오류: {req_err}")
        results_summary.append({keyword_to_search: "요청 오류"})
        
    except Exception as e:
        print(f"  → ❌ 알 수 없는 오류: {e}")
        results_summary.append({keyword_to_search: "알 수 없는 오류"})

    time.sleep(3)

print(f"\n📦 모든 쿼리 실행 완료.")
# --- 총 합계 출력 ---
print(f"📊 전체 추정 합계 (각 쿼리 결과의 단순 합): {total_repos_aggregated}개")
print("   (참고: 이 합계는 여러 키워드에 해당하는 저장소가 중복 계산될 수 있습니다.)")


if results_summary:
    processed_results_for_df = []
    for entry in results_summary:
        for keyword, count_or_error in entry.items():
            processed_results_for_df.append({'Keyword': keyword, 'Repository Count': count_or_error})
    df_results = pd.DataFrame(processed_results_for_df)
    
    print("\n--- 개별 키워드 검색 결과 요약 ---")
    print(df_results.to_string())
else:
    print("\n결과가 없습니다.")

총 12개의 키워드 쿼리를 실행합니다.
사용될 기본 필터: in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31

참고: 키워드 내 큰따옴표가 제거되어, 여러 단어 키워드는 AND 조건으로 검색됩니다.
(예: 'artificial intelligence'는 'artificial' AND 'intelligence'로 검색)

(1/12) 실행 중인 쿼리: "artificial intelligence" OR "computer vision" OR "deep learning" OR "machine learning" OR "image recognition" in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31
  → 결과: 30577개 저장소
(2/12) 실행 중인 쿼리: "natural language processing" OR "nlp" OR "language model" OR "speech recognition" in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31
  → 결과: 13555개 저장소
(3/12) 실행 중인 쿼리: agentic AI in:name,description,readme language:python created:2013-01-01..2024-12-31 stars:>10 pushed:2020-01-01..2024-12-31
  → 결과: 593개 저장소
(4/12) 실행 중인 쿼리: causal AI in:name,description,readme language:python created:201

In [39]:
print(f"\n📦 모든 쿼리 실행 완료.")
# --- 총 합계 출력 ---
print(f"📊 전체 추정 합계 (각 쿼리 결과의 단순 합): {total_repos_aggregated}개")
print("   (참고: 이 합계는 여러 키워드에 해당하는 저장소가 중복 계산될 수 있습니다.)")


if results_summary:
    processed_results_for_df = []
    for entry in results_summary:
        for keyword, count_or_error in entry.items():
            processed_results_for_df.append({'Keyword': keyword, 'Repository Count': count_or_error})
    df_results = pd.DataFrame(processed_results_for_df)
    
    print("\n--- 개별 키워드 검색 결과 요약 ---")
    print(df_results.to_string())
else:
    print("\n결과가 없습니다.")


📦 모든 쿼리 실행 완료.
📊 전체 추정 합계 (각 쿼리 결과의 단순 합): 47746개
   (참고: 이 합계는 여러 키워드에 해당하는 저장소가 중복 계산될 수 있습니다.)

--- 개별 키워드 검색 결과 요약 ---
                                                                                                           Keyword  Repository Count
0   "artificial intelligence" OR "computer vision" OR "deep learning" OR "machine learning" OR "image recognition"             30577
1                               "natural language processing" OR "nlp" OR "language model" OR "speech recognition"             13555
2                                                                                                       agentic AI               593
3                                                                                                        causal AI               172
4                                                                                                   explainable AI                64
5                                                                             